# FutureTraffic Colab 실행

위에서부터 코드 셀 왼쪽의 **▶ 버튼**을 하나씩 누릅니다.  
처음에는 연결 확인을 위해 5,000단계만 학습합니다.


## 1. Google Drive 연결

실행 후 나타나는 **Connect to Google Drive** 버튼을 누르고, 계정을 선택한 다음 **계속**을 누릅니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 데이터 압축파일 확인

Google Drive의 `내 드라이브/FutureTraffic/` 안에 `FutureTraffic_colab_data.zip`이 있어야 합니다.


In [ ]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/FutureTraffic')
DATA_ZIP = DRIVE_PROJECT / 'FutureTraffic_colab_data.zip'
assert DATA_ZIP.exists(), f'파일을 찾지 못했습니다: {DATA_ZIP}'
print('데이터 확인:', DATA_ZIP)


## 3. GitHub 코드 받기

저장소가 없으면 복제하고, 이미 있으면 최신 코드로 갱신합니다.


In [ ]:
import subprocess

REPO = Path('/content/FutureTraffic')
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/wkdus0608/FutureTraffic.git', str(REPO)],
        check=True,
    )
print('코드 준비 완료:', REPO)


## 4. 필요한 프로그램 설치

처음 실행할 때 몇 분 걸릴 수 있습니다. 마지막에 오류 없이 끝나면 정상입니다.


In [ ]:
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
    check=True,
)
print('설치 완료')


## 5. SUMO 데이터 풀기

Drive의 압축파일을 Colab 임시 저장공간에 풉니다.


In [ ]:
from zipfile import ZipFile

with ZipFile(DATA_ZIP) as archive:
    archive.extractall(REPO)

SCENARIO = REPO / 'data/processed/20220810_0500'
required = [
    SCENARIO / 'network.net.xml',
    SCENARIO / 'traffic_20220810_0500.rou.xml',
    SCENARIO / 'scenario.sumocfg',
]
assert all(path.exists() for path in required), '압축파일 안의 SUMO 파일을 확인하세요.'
print('SUMO 데이터 준비 완료')
for path in required:
    print('-', path.name)


## 6. SUMO-RL 연결 확인

`관측값 개수: 45`, `선택 가능한 녹색 신호: 4개`가 나오면 성공입니다.


In [ ]:
subprocess.run(
    ['python', 'src/check_rl_environment.py', '--steps', '10'],
    cwd=REPO,
    check=True,
)


## 7. DQN 시험 학습

처음에는 5,000단계로 전체 과정만 확인합니다. 이후 `TIMESTEPS`를 늘릴 수 있습니다.


In [ ]:
TIMESTEPS = 5000  # @param {type:"integer"}

subprocess.run(
    ['python', 'src/train_dqn.py', '--timesteps', str(TIMESTEPS)],
    cwd=REPO,
    check=True,
)


## 8. 고정신호·무작위·DQN 평가

같은 차량과 seed로 세 방법을 비교합니다.


In [ ]:
subprocess.run(
    ['python', 'src/evaluate_policies.py', '--seeds', '42'],
    cwd=REPO,
    check=True,
)


## 9. 모델과 결과를 Drive에 저장

Colab 연결이 종료되어도 결과가 남도록 Drive로 복사합니다.


In [ ]:
from datetime import datetime
import shutil

run_name = datetime.now().strftime('%Y%m%d_%H%M%S')
destination = DRIVE_PROJECT / 'runs' / run_name
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPO / 'results', destination)
print('Drive 저장 완료:', destination)


## 10. 평가표 확인


In [ ]:
from IPython.display import Markdown, display

report = REPO / 'results/evaluation_20220810_0500/report.md'
display(Markdown(report.read_text(encoding='utf-8')))
